# Simulations of a Geobody

<!-- SUMMARY: Simulations of a Geobody -->

<!-- CATEGORY: Case Studies -->

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

import gstlearn as gl
import gstlearn.plot as gp
import gstlearn.plot3D as gop
import matplotlib.pyplot as plt

In [ ]:
ndim = 3
gl.defineDefaultSpace(gl.ESpaceType.RN, ndim)

In [ ]:
# 3D cubic grid
size = 100
radius1 = 15 # Hole radius
radius2 = 30 # Enveloppe radius
corner = [40,50,50]

x0, y0, z0 = (40, 55, 45) # Hole gravity center
x, y, z = np.mgrid[0:size:1, 0:size:1, 0:size:1]
r = np.sqrt((x - x0)**2 + (y - y0)**2 + ((z - z0)**2)/3)
s = np.zeros_like(r)
s[r > radius1] = 1
x0, y0, z0 = (50, 50, 50) # Enveloppe gravity center
x, y, z = np.mgrid[0:size:1, 0:size:1, 0:size:1]
r = np.sqrt((x - x0)**2 + (y - y0)**2 + ((z - z0)**2)/2)
s[r > radius2] = 0
grid0 = gl.DbGrid.create(nx = [size,size,size])
grid0.addSelection(s.flatten(), "Sel_Body")
ax = grid0.plot("Sel_Body",posX=0,posY=1,corner=corner)

In [ ]:
# Extend the 3D grid
nx = 140
ny = 120
nz = 100
grid = gl.DbGrid.create(nx = [nx,ny,nz], x0 = [-20,-10,0])
gl.migrate(grid0, grid, "Sel_Body", namconv = gl.NamingConvention("Sel0", False))
s = grid['Sel0'].flatten()
grid['Sel_Body'] = np.nan_to_num(s)

model1 = gl.Model.createFromParam(gl.ECov.MATERN,param=0.8,range=5,space=gl.SpaceRN(3))
model1.setMean(10)
model2 = gl.Model.createFromParam(gl.ECov.MATERN,param=0.8,range=10,space=gl.SpaceRN(3))
model2.setMean(8)

grid.setLocator("Sel_Body", gl.ELoc.SEL)
gl.simtub(None,dbout=grid,model=model1)
grid.addSelection(1 - grid['Sel_Body'].flatten(), "Sel_Out")
gl.simtub(None,dbout=grid,model=model2)

grid["Val"] = grid["Simu"] + grid["Simu.1"]
grid.display()

res = grid["Val"].reshape(grid.getNXs())
data = [gop.SliceOnDbGrid(grid,"Val",0,corner[0]),
        gop.SliceOnDbGrid(grid,"Val",2,corner[1]),
        gop.SliceOnDbGrid(grid,"Val",1,corner[2])
       ]
fig1 = go.Figure(data=data)
f = fig1.show()

In [ ]:
# Extract wells
rng = np.random.default_rng(25)
nwells = 30
iw = 0
wellc = np.arange(1, nwells+1)
wellx = rng.integers(nz, size=nwells)
welly = rng.integers(nz, size=nwells)
wellz = np.arange(0, nz)
pile = gl.VectorDouble()
iuids = grid.getUIDs(["Sel_Body", "Val"])
s = gl.VectorDouble()
v = gl.VectorDouble()
for iw in range(nwells):
    indg = [wellx[iw], welly[iw], 0]
    grid.getGridPileInPlace(iuids[0], indg, 2, pile)
    s = gl.VH.concatenate(s, pile)
    grid.getGridPileInPlace(iuids[1], indg, 2, pile)
    v = gl.VH.concatenate(v, pile)

wellc = np.repeat(wellx, nz)
wellx = np.repeat(wellx, nz)
welly = np.repeat(welly, nz)
wellz = np.tile(wellz, nwells)

data = gl.Db()
data.addColumns(wellx, "x1",       gl.ELoc.X, 0)
data.addColumns(welly, "x2",       gl.ELoc.X, 1)
data.addColumns(wellz, "x3",       gl.ELoc.X, 3)
data.addColumns(wellc, "code",     gl.ELoc.C, 0)
data.addColumns(v,     "Val",      gl.ELoc.Z, 0)
data.addColumns(s,     "Sel_Body")
data.display()

fig = px.scatter_3d(data.toTL(), x='x1', y='x2', z='x3', color='Val')
fig.show()